# 10 · Bounded duration and ConvNeXt-Tiny pilots

Ask two focused optimisation questions after the clean baseline: whether 24 epochs improve the fixed 12-epoch endpoint, and whether ConvNeXt-Tiny improves the matching B0 pilot. Keep optimisation evidence separate from independent outer-fold evaluation.

Read the stages in numerical order. Training uses separate, explicitly invoked runners; executing this notebook only loads development data and saved artifacts.

Original notebooks, images, checkpoints and results remain available. The reserved test is outside this workflow.

**Original methods and figure reference:** [10_image_architecture_comparison.ipynb](../Subset/10_image_architecture_comparison.ipynb), [11_image_augmentation.ipynb](../Subset/11_image_augmentation.ipynb), [12_image_lighting.ipynb](../Subset/12_image_lighting.ipynb). Read [method coverage](METHOD_COVERAGE.md) for retained methods and declared adaptations.

## Load the frozen clean development dataset

In [1]:
NOTEBOOK_STAGE = '10_bounded_image_pilots'
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from melanoma_detection.development_dataset import load_development_dataset, load_preserved_inner_memberships
from melanoma_detection.dedup_workflow import load_context, latest_evaluation, load_clean_scores, load_image_report, load_uncertainty_report, read_json, project_path, DIRECTORY, METADATA_SUITE, UNCERTAINTY
get_ipython().run_line_magic("matplotlib", "inline")
MANIFEST = "data/subset/processed/dedup_v1/dataset_manifest.json"
data, dataset_identity = load_development_dataset(ROOT, dataset_manifest=MANIFEST)
context = load_context(ROOT)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
FIGURE_DIR = ROOT / DIRECTORY / "figures" / NOTEBOOK_STAGE
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def finish_figure(figure, filename):
    if not filename or Path(filename).name != filename:
        raise ValueError("Figure name must be a simple stem.")
    figure.savefig(FIGURE_DIR / (filename + ".png"), dpi=180, bbox_inches="tight")
    figure.savefig(FIGURE_DIR / (filename + ".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(figure)

_score_bundle = None
def score_bundle():
    global _score_bundle
    if _score_bundle is None:
        _score_bundle = load_clean_scores(ROOT)
    return _score_bundle
display(pd.DataFrame([{"Images": len(data), "Malignant images": int(data.target.sum()),
    "Patients": data.patient_id.nunique(), "Malignant patients": data.loc[data.target.eq(1), "patient_id"].nunique()}]))
display(Markdown("**Dataset:** `dedup_v1` · original patient folds · development records only. This notebook reads saved results and starts no training."))

def show_saved_table(relative, limit=30):
    path = project_path(ROOT, relative)
    if not path.exists():
        display(Markdown("**Pending:** " + relative + ". No results are inferred from unfinished runs."))
        return pd.DataFrame()
    table = pd.read_csv(path)
    display(table.head(limit))
    if len(table) > limit:
        display(Markdown(f"Showing {limit} of {len(table)} rows; the complete CSV is saved."))
    return table

def comparison_table():
    evaluation = latest_evaluation(ROOT)
    if "output_directory" not in evaluation:
        display(Markdown("**Pending:** clean evaluation has not been generated."))
        return pd.DataFrame()
    path = project_path(ROOT, evaluation["output_directory"] + "/model_comparison.csv")
    return pd.read_csv(path)

def model_chart(table, title):
    if table.empty:
        display(Markdown("**Pending:** no verified scores are available for this stage."))
        return
    table = table.sort_values("average_precision")
    fig, ax = plt.subplots(figsize=(9, max(3.2, .38 * len(table) + 1.5)), layout="constrained")
    ax.barh(table.model, table.average_precision, color="#276b91")
    ax.set(title=title, xlabel="Unweighted mean outer-fold average precision", xlim=(0, max(.1, float(table.average_precision.max()) * 1.25)))
    finish_figure(fig, "model_comparison_ap")

def pilot_report(run_id):
    directory = ROOT / "results/image_d1_runs" / run_id
    path = directory / "summary.json"
    summary = read_json(path) if path.exists() else {"status": "pending"}
    if summary.get("status") == "complete":
        manifest = read_json(directory / "run_manifest.json")
        if manifest["dataset_identity"] != dataset_identity:
            raise ValueError("Pilot belongs to another dataset.")
        from melanoma_detection.development_dataset import sha256_file
        for relative, digest in summary["artifact_sha256"].items():
            artifact = (directory / relative).resolve()
            if not artifact.is_relative_to(directory.resolve()) or sha256_file(artifact) != digest:
                raise ValueError("Completed pilot artifact changed.")
    return directory, summary


,Images,Malignant images,Patients,Malignant patients
0,6706,127,422,88


**Dataset:** `dedup_v1` · original patient folds · development records only. This notebook reads saved results and starts no training.

## Controls, fitting scope and deferred work

Both pilots use outer fold 0’s training pool and its preserved three inner patient folds. Common settings are retained image IDs, 224-pixel RGB preparation, weak augmentation, effective batch 64 and the same ranking objective. Every distinct fit starts from original pretrained weights. The original 30-epoch scheduler remains fixed while the new cap is 24. DenseNet sweeps, full alternative-architecture OOF, stronger augmentation, CLAHE and resolution changes remain deferred.

In [2]:
from melanoma_detection import dedup_notebook_images as image_docs
inner = load_preserved_inner_memberships(ROOT, dataset_identity, 0)
display(inner.groupby("inner_fold").agg(images=("image_name", "size"), patients=("patient_id", "nunique"), malignant_images=("target", "sum")))
display(pd.DataFrame([{"Pilot": "B0 duration", "Fits": 3, "Epochs": 72, "Endpoints": "12 and 24", "Recipe": "clean baseline’s selected outer-pool recipe"}, {"Pilot": "ConvNeXt-Tiny", "Fits": 6, "Epochs": 90, "Endpoints": "12 and 24", "Recipe": "four six-epoch screens on inner 0, then selected confirmations"}]))

,images,patients,malignant_images
inner_fold,,,
0,1834,113,33
1,1704,111,35
2,1824,114,35


,Pilot,Fits,Epochs,Endpoints,Recipe
0,B0 duration,3,72,12 and 24,clean baseline’s selected outer-pool recipe
1,ConvNeXt-Tiny,6,90,12 and 24,"four six-epoch screens on inner 0, then select..."


## Completion ledger and selected recipes

ConvNeXt-Tiny changes network design while preserving preparation. Its four six-epoch recipes screen on inner 0; the selected clean trajectory continues to 24 and the same recipe fits inner 1 and 2. A completed family is admitted only after endpoint predictions, memberships and receipt hashes verify. Missing or budget-stopped families do not lose a comparison.

In [3]:
pilots = image_docs.load_pilot_bundle(pilot_report)
image_docs.pilot_screening(pilots)

,family,run_id,status,completed_inner_folds,stop_reason
0,B0,b0_duration_dedup_v1,pending,[],None
1,ConvNeXt-Tiny,convnext_duration_dedup_v1,pending,[],None


**Pending:** No complete three-inner-fold pilot is available; remaining sections wait for matching committed exports.

## Learning over epochs

Read completed epoch horizontally and AP/AUC, log loss and Brier vertically. Each line is a family and preserved inner split; solid B0 and dashed ConvNeXt lines share the same 12/24 guides. AP and AUC are higher-is-better; log loss and Brier are lower-is-better. These curves describe selected inner recipes, not independent outer performance.

In [4]:
image_docs.pilot_learning(pilots, finish_figure)

## Matching fixed endpoints and best-in-cap metrics

The same fresh trajectories supply frozen epoch-12 and epoch-24 snapshots. The left panel compares predeclared fixed endpoints; the right reports AP-selected best epochs within each cap. Best-in-cap metrics reuse validation labels for selection and are more optimistic. Mean inner-fold AP is primary, with AUC and score-quality metrics alongside it. The cap remains 24 even if the best epoch reaches its boundary.

In [5]:
image_docs.pilot_endpoint_metrics(pilots, finish_figure)

## ROC, precision–recall and descriptive reliability

Only completed fixed-24 exports on exactly matching retained inner image/patient records enter these figures. ROC plots recall against false alarms; PR plots precision against recall, with the malignant-image fraction as a reference. Equal-count reliability compares mean raw score with observed malignancy and shows bin support. Recipe selection already used these patients, so favourable curves are exploratory.

In [6]:
image_docs.pilot_matching_curves(pilots, finish_figure)

## Fixed-cutoff errors

Flag raw scores at the declared 0.5 cutoff; no inner-validation cutoff is selected here. Confusion rows are recorded benign/malignant labels and columns are predicted decisions. FN are missed malignancies; FP are benign flags. Recall, specificity, precision and flagged counts complement ranking; undefined ratios remain missing. Different class-weighting and network score scales make this a descriptive scenario.

In [7]:
image_docs.pilot_decisions(pilots, finish_figure)

## Recorded training time and GPU memory

The bars compare recorded fitting minutes and peak allocated GPU GiB. The table also records reserved memory, distinct fits and epochs. ConvNeXt includes unsuccessful recipe screens in its workload. Peak memory and fitting minutes describe this machine and protocol; they cannot establish a network’s general speed or justify ignoring matching prediction quality.

In [8]:
image_docs.pilot_cost(pilots, finish_figure)

## Paired patient intervals and support

Whole-patient draws within the three inner folds serve both sides of each 24-minus-12 or ConvNeXt-minus-B0 comparison. Positive AP/AUC differences favour the candidate. The interval table records requested, valid and single-class-rejected draws, and the support table exposes malignant-image counts. These conditional intervals omit retraining and selection variability; best-in-cap comparisons remain selected evidence.

In [9]:
image_docs.pilot_interval_support(pilots, finish_figure)

## Qualified interpretation and next step

Duration and architecture effects are kept separate from the clean baseline. A positive fixed-endpoint AP change can motivate another procedure but cannot automatically promote it. An incomplete family supplies no comparison result; neither a favourable epoch nor a favourable raw cutoff proves generalisation.

In [10]:
display(Markdown("**Interpretation:** a promising pilot is a candidate procedure. Promotion requires selection repeated within every outer-training pool and a separate five-fold evaluation. Full DenseNet/ConvNeXt OOF, stronger augmentation, CLAHE, resolution changes and fusion remain deferred."))

**Interpretation:** a promising pilot is a candidate procedure. Promotion requires selection repeated within every outer-training pool and a separate five-fold evaluation. Full DenseNet/ConvNeXt OOF, stronger augmentation, CLAHE, resolution changes and fusion remain deferred.

## Interpretation boundary

These are development results. Patient grouping protects separation between fitting and validation; bootstrap intervals condition on saved models. Sparse subgroups and selected inner-validation pilots cannot establish a final winner. No reserved-test conclusion is made.